In [24]:
import re
import pandas as pd
import time
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns


### Reading in the Data

In [25]:
race = 'TOR330'

In [26]:
TOR330_dem = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_dem.xlsx')

In [27]:
TOR330_dem['Bib'][(TOR330_dem['Year'] == 2022) &
               (TOR330_dem['Retired'] == 'Bosses')].nunique()

0

In [28]:
TOR330_dem.groupby( ['Year','Status1'] )['Status1' ].count()

Year  Status1                     
2021  DNF                             180
      Finished at Courmayeur          431
2022  DNF                             250
      Finished at Bosses               88
      Finished at Courmayeur          408
      Finished at Rifugio Frassati    101
2023  DNF                             407
      Finished at Courmayeur          622
2024  DNF                             368
      Finished at Courmayeur          533
Name: Status1, dtype: int64

In [ ]:
# lifebase_cut_offs_df = pd.read_excel(f'{race} Data/4. TOR330 Timetable Data/{race}_lifebase_cut_offs_df.xlsx')

# # # # Convert integer seconds to timedelta
# lifebase_cut_offs_df['Lifebase Duration'] = pd.to_timedelta(lifebase_cut_offs_df['Lifebase Duration_seconds'], unit='s')

# # # # Convert integer seconds to timedelta
# lifebase_cut_offs_df['Running Total Lifebase Duration'] = pd.to_timedelta(lifebase_cut_offs_df['Running Total Lifebase Duration_seconds'], unit='s')

# lifebase_cut_offs_df

In [29]:
checkpoints_bib_df = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/Clean 100x100trail Data/{race}_checkpoints_bib_df.xlsx')

In [30]:
TOR330_dem.groupby(['Retired_Stage','Retired', 'Retired_Detail'])['PK'].count()

Retired_Stage  Retired                  Retired_Detail         
DNS            Start                    Start                      136
Stage 1        Baite Youlaz             Baite Youlaz                 2
               La Thuile                La Thuile                   41
               Planaval                 Planaval                    29
               Rifugio Deffeyes         Rifugio Deffeyes            57
               Valgrisenche             Valgrisenche IN              0
                                        Valgrisenche OUT             0
Stage 2        Chalet Epee              Chalet Epee                 30
               Cogne                    Cogne IN                     0
                                        Cogne OUT                    0
               Eaux Rousse              Eaux Rousse                182
               Rhemes-Notre-Dame        Rhemes-Notre-Dame          130
               Rifugio Sella            Rifugio Sella               23
Stage 3      

In [ ]:
all_aid_station_bib_df = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/Clean 100x100trail Data/{race}_all_aid_station_bib_df.xlsx')

In [ ]:
lifebase_bib_df = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/Clean 100x100trail Data/{race}_lifebase_bib_df.xlsx')

In [ ]:
lifebase_bib_df = lifebase_bib_df.merge(
        TOR330_dem[['PK', 'Name','Status1']],
        on=['PK'],
        how='left')
lifebase_bib_df.head()

In [ ]:
# no_banking_df = lifebase_bib_df[(lifebase_bib_df['Status1'].str.contains( 'Courmayeur')) &
#                 (lifebase_bib_df['Lifebase'] != 'START') &
#                 (lifebase_bib_df['Banking Time_seconds'] <0)]

# no_banking_df['Banking Time_seconds'] = no_banking_df['Banking Time_seconds'].astype(int)
# # Apply function to the column
# no_banking_df['Banking Time'] =pd.to_timedelta(no_banking_df['Banking Time_seconds'], unit='s').astype(str)
no_banking_df['Running Total Duration'] =pd.to_timedelta(no_banking_df['Running Total Duration_seconds'], unit='s').astype(str)

for unique_name in no_banking_df['Name']:
    print(no_banking_df[['Name','Year', 'Race', 'Lifebase','Running Total Duration', 'Banking Time']][no_banking_df['Name'] == unique_name])
    print('*'*40)



In [ ]:
# # making duration hours 
# datasets = [checkpoints_bib_df, lifebase_bib_df, all_aid_station_bib_df,  TOR330_dem]
# for df in datasets:
#     df['Duration_hours'] = df['Duration_seconds']/ 3600 

In [ ]:
all_aid_station_bib_df

### Getting Average and Median time for lifebases

In [7]:
Stage1 = [ 'Start', 'Baite Youlaz', 'La Thuile', 'Rifugio Deffeyes',\
          'Planaval', 'Valgrisenche IN']
Stage2 = [ 'Valgrisenche OUT', 'Chalet Epee',\
          'Rhemes-Notre-Dame', 'Eaux Rousse', 'Rifugio Sella', 'Cogne IN']
Stage3 = [  'Cogne OUT', 'Goilles', 'Rifugio Dondena', 'Chardonney', 'Pontboset','Donnas IN']
Stage4 = [  'Donnas OUT', 'Perloz', 'Sassa', 'Rifugio Coda', \
          'Rifugio della Barma', 'Lago Chiaro', 'Col della Vecchia',\
          'Niel La Gruba', 'Loo', 'Gressoney IN']
Stage5 = [  'Gressoney OUT', 'Rifugio Alpenzu', 'Champoluc' ,\
          'Rifugio Grand Tournalin', 'Valtournenche IN']
Stage6 = [   'Valtournenche OUT', 'Rifugio Barmasse', 'Vareton',\
          'Rifugio Magià', 'Rifugio Cuney', 'Bivacco R. Clermont', 'Oyace', \
          'Bruson Arp', 'Ollomont IN']    
Stage7 = [ 'Ollomont OUT', 'Rifugio Champillon', 'Ponteille Desot',\
          'Bosses', 'Rifugio Frassati', 'Pas Entre Deux Sauts',\
          'Monte de la Saxe', 'FINISH']


stages =[ Stage1, Stage2, Stage3, Stage4, Stage5, Stage6, Stage7]
stages_str =[ 'Stage 1', 'Stage 2', 'Stage 3', 'Stage 4', 'Stage 5', 'Stage 6', 'Stage 7']
lifebase_time_spent = ['Valgrisenche OUT','Cogne OUT','Donnas OUT','Gressoney OUT','Valtournenche OUT','Ollomont OUT']


In [8]:
def creating_time_stats(df, column, category_order):
    
    df_merge = df.merge(
        TOR330_dem[['PK', 'Finish Category']],
        on=['PK'],
        how='left')


    df_merge['Duration_seconds'][df_merge['Duration_seconds']  <= 0] = np.nan

    stats_df = df_merge.groupby(['Finish Category', column])['Duration_seconds'].describe().reset_index(drop = False)


    stats_df[[ 'mean', 'std', 'min', '25%',
           '50%', '75%', 'max']] = stats_df[[ 'mean', 'std', 'min', '25%',
           '50%', '75%', 'max']].round(0)

    # Set 'Finish Category' as a categorical column with the defined order
    stats_df[column] = pd.Categorical(
        stats_df[column],
        categories=category_order,
        ordered=True)
    
    stats_df = stats_df.sort_values(by=column, ascending = True)
    
    

    for finish_category in df_merge['Finish Category'].unique():
        print(finish_category)
        stats_df.loc[
                (stats_df['Finish Category'] == finish_category), 'running_total_mean_seconds'
            ] =     stats_df.loc[
                (stats_df['Finish Category'] == finish_category), 'mean'
            ].cumsum()


        stats_df.loc[
                (stats_df['Finish Category'] == finish_category), 'running_total_median_seconds'
            ] =     stats_df.loc[
                (stats_df['Finish Category'] == finish_category), '50%'
            ].cumsum()


    stats_df = stats_df[['Finish Category', column,
            'count', 'mean', '50%', 'std', 'min', 'max', 
            'running_total_mean_seconds', 'running_total_median_seconds']]


    stats_df = stats_df.rename(columns={'count': f'Count_Finish_Category_{column}_seconds',
                                      'mean': f'Mean_Finish_Category_{column}_seconds',
                                      'std': f'STD_Finish_Category{column}t_seconds',
                                      '50%': f'Median_Finish_Category_{column}_seconds',
                                      'min': f'Min_Finish_Category_{column}_seconds', 
                                      'max': f'Max_Finish_Category_{column}_seconds'})
    stats_df = stats_df[stats_df[column] != 'Start']

    
    
    
    for stage,  stage_str in zip(stages, stages_str):
        stats_df.loc[stats_df[column].isin(stage), 'Stage'] = f'{stage_str}'
        
    for lifebase in lifebase_time_spent:
        lifebase_split = lifebase.split(' OUT')[0] 
        stats_df.loc[stats_df[column] == lifebase, 'Stage'] = f'Time Spent in {lifebase_split}'


    stats_df.to_excel(f'{race} Data/5. Clean Data for Data Visualisation/Schedule Splits/{race}_{column}_duration_mean_median_stats_df.xlsx', index = False)

#     print(stats_df[stats_df['Finish Category'] == 'Sub-130'])
    return stats_df

In [ ]:
lifebase_stats_df = creating_time_stats(df = lifebase_bib_df, 
                    column = 'Lifebase', 
                    category_order = ['Start','Valgrisenche IN','Valgrisenche OUT',
                            'Cogne IN',  'Cogne OUT',
                            'Donnas IN', 'Donnas OUT', 
                            'Gressoney IN','Gressoney OUT',
                            'Valtournenche IN','Valtournenche OUT', 
                            'Ollomont IN','Ollomont OUT',
                            'FINISH'])

lifebase_stats_df

In [ ]:
# # # Convert integer seconds to timedelta
lifebase_stats_df['Median_Finish_Category_Lifebase'] = pd.to_timedelta(lifebase_stats_df['Median_Finish_Category_Lifebase_seconds'], unit='s')


lifebase_stats_df[['Finish Category','Lifebase', 'Median_Finish_Category_Lifebase']][lifebase_stats_df['Finish Category'] == 'Sub-130']

In [9]:
checkpoints_stats_df = creating_time_stats(df = checkpoints_bib_df, 
                    column = 'Checkpoint', 
                    category_order = ['La Thuile', 'Valgrisenche IN', 'Valgrisenche OUT',
                                       'Eaux Rousse', 'Cogne IN', 'Cogne OUT', 'Donnas IN', 'Donnas OUT',
                                       'Rifugio della Barma', 'Niel La Gruba', 'Gressoney IN',
                                       'Gressoney OUT', 'Champoluc', 'Valtournenche IN',
                                       'Valtournenche OUT', 'Oyace', 'Ollomont IN', 'Ollomont OUT',
                                       'FINISH'])

checkpoints_stats_df

C:\Users\Karina\AppData\Local\Temp\ipykernel_13980\1727915605.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_merge['Duration_seconds'][df_merge['Duration_seconds']  <= 0] = np.nan


Sub-70
Sub-80
Sub-90
Sub-100
Sub-110
Sub-120
Sub-130
Sub-140
Sub-150
Over-150
DNF
nan


,Finish Category,Checkpoint,Count_Finish_Category_Checkpoint_seconds,Mean_Finish_Category_Checkpoint_seconds,Median_Finish_Category_Checkpoint_seconds,STD_Finish_CategoryCheckpointt_seconds,Min_Finish_Category_Checkpoint_seconds,Max_Finish_Category_Checkpoint_seconds,running_total_mean_seconds,running_total_median_seconds,Stage
109,Sub-130,La Thuile,3806.0,12404.0,12406.0,1465.0,8913.0,21942.0,12404.0,12406.0,Stage 1
49,Sub-100,La Thuile,678.0,10347.0,10139.0,797.0,8291.0,17390.0,10347.0,10139.0,Stage 1
69,Sub-110,La Thuile,1039.0,10909.0,10701.0,1291.0,8877.0,19770.0,10909.0,10701.0,Stage 1
89,Sub-120,La Thuile,1676.0,11899.0,11575.0,1648.0,9291.0,21590.0,11899.0,11575.0,Stage 1
29,Over-150,La Thuile,65.0,16057.0,16072.0,2606.0,13981.0,23751.0,16057.0,16072.0,Stage 1
...,...,...,...,...,...,...,...,...,...,...,...
135,Sub-140,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
155,Sub-150,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
175,Sub-70,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
195,Sub-80,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# all_aid_station_bib_stats_df = creating_time_stats(df = all_aid_station_bib_df, 
#                     column = 'Aid Station', 
#                     category_order = ['Baite Youlaz', 'La Thuile', 'Rifugio Deffeyes',
#                                    'Planaval', 'Valgrisenche IN', 'Valgrisenche OUT', 'Chalet Epee',
#                                    'Rhemes-Notre-Dame', 'Eaux Rousse', 'Rifugio Sella', 'Cogne IN',
#                                    'Cogne OUT', 'Goilles', 'Rifugio Dondena', 'Chardonney', 'Pontboset',
#                                    'Donnas IN', 'Donnas OUT', 'Perloz', 'Sassa', 'Rifugio Coda',
#                                    'Rifugio della Barma', 'Lago Chiaro', 'Col della Vecchia',
#                                    'Niel La Gruba', 'Loo', 'Gressoney IN', 'Gressoney OUT',
#                                    'Rifugio Alpenzu', 'Champoluc', 'Rifugio Grand Tournalin',
#                                    'Valtournenche IN', 'Valtournenche OUT', 'Rifugio Barmasse', 'Vareton',
#                                    'Rifugio Magià', 'Rifugio Cuney', 'Bivacco R. Clermont', 'Oyace',
#                                    'Bruson Arp', 'Ollomont IN', 'Ollomont OUT', 'Rifugio Champillon',
#                                    'Ponteille Desot', 'Bosses', 'Rifugio Frassati', 'Pas Entre Deux Sauts',
#                                    'Monte de la Saxe', 'FINISH'])

# all_aid_station_bib_stats_df

,PK,Year,Race,Bib,Wave,Checkpoint,Stage,Timestamp,Duration_seconds,Running Total Duration_seconds,Banking Time_seconds,Missed Lifebase Allocated Time,Banking Time_hours,Running Total Duration_hours,Banking Time_seconds_hours,Running Total Duration_seconds_hours
57320,TOR330_2023_1539,2023,TOR330,1539,Wave2,Start,Stage 1,2023-09-10 12:00:00,NaN,0.0,NaN,Within Allocated Time,NaT,0 days 00:00:00,NaN,0.000000
57321,TOR330_2023_1539,2023,TOR330,1539,Wave2,La Thuile,Stage 1,2023-09-10 16:18:27,15507.0,15507.0,4293.0,Within Allocated Time,0 days 01:11:33,0 days 04:18:27,1.192500,4.307500
57322,TOR330_2023_1539,2023,TOR330,1539,Wave2,Valgrisenche IN,Stage 1,2023-09-11 01:33:52,33325.0,48832.0,19568.0,Within Allocated Time,0 days 05:26:08,0 days 13:33:52,5.435556,13.564444
57323,TOR330_2023_1539,2023,TOR330,1539,Wave2,Valgrisenche OUT,Time Spent in Valgrisenche,2023-09-11 03:15:09,6077.0,54909.0,20691.0,Within Allocated Time,0 days 05:44:51,0 days 15:15:09,5.747500,15.252500
57324,TOR330_2023_1539,2023,TOR330,1539,Wave2,Eaux Rousse,Stage 2,2023-09-11 14:17:10,39721.0,94630.0,25970.0,Within Allocated Time,0 days 07:12:50,1 days 02:17:10,7.213889,26.286111
57325,TOR330_2023_1539,2023,TOR330,1539,Wave2,Cogne IN,Stage 2,2023-09-11 22:48:54,30704.0,125334.0,25866.0,Outside Allocated Time,0 days 07:11:06,1 days 10:48:54,7.185000,34.815000
57326,TOR330_2023_1539,2023,TOR330,1539,Wave2,Cogne OUT,Time Spent in Cogne,2023-09-12 04:25:38,20204.0,145538.0,12862.0,Outside Allocated Time,0 days 03:34:22,1 days 16:25:38,3.572778,40.427222
57327,TOR330_2023_1539,2023,TOR330,1539,Wave2,Donnas IN,Stage 3,2023-09-12 15:15:15,38977.0,184515.0,38685.0,Within Allocated Time,0 days 10:44:45,2 days 03:15:15,10.745833,51.254167
57328,TOR330_2023_1539,2023,TOR330,1539,Wave2,Donnas OUT,Time Spent in Donnas,2023-09-12 19:29:40,15265.0,199780.0,30620.0,Outside Allocated Time,0 days 08:30:20,2 days 07:29:40,8.505556,55.494444
57329,TOR330_2023_1539,2023,TOR330,1539,Wave2,Rifugio della Barma,Stage 4,2023-09-13 06:17:45,38885.0,238665.0,31335.0,Within Allocated Time,0 days 08:42:15,2 days 18:17:45,8.704167,66.295833


### Who ran too easy or hard at the start?

In [ ]:
# sub_TOR330_dem_bib_list = list(TOR330_dem['PK'][TOR330_dem['Finish Category'] == 'Sub-130'].unique())

# new_bib_list = []
# for bib in sub_TOR330_dem_bib_list:
#     df = lifebase_bib_df[
#                 ((lifebase_bib_df['Duration_hours']> 14) &
#                 (lifebase_bib_df['Lifebase'] == 'Valgrisenche IN') )
#     &
#                 (lifebase_bib_df['PK']== bib)
#                ]
    
#     new_bib_list.append(df)
# df= pd.concat(new_bib_list)
# pk_unique = list(df['PK'].unique())

# for pk in pk_unique:
#     print(lifebase_bib_df[['PK','Lifebase', 'Timestamp', 'Duration_hours']][(lifebase_bib_df['PK'] == pk)  ], '\n', '*'*40,  '\n',)
#     print(TOR330_dem[['PK', 'Finish Category', 'Duration_hours']][(TOR330_dem['PK'] == pk)  ], '\n','\n','\n','\n', '*'*40,  '\n',)

In [ ]:
# sub_TOR330_dem_bib_list = list(TOR330_dem['PK'][TOR330_dem['Finish Category'] == 'Sub-140'].unique())

# new_bib_list = []
# for bib in sub_TOR330_dem_bib_list:
#     df = lifebase_bib_df[
#                 ((lifebase_bib_df['Duration_hours']< 10) &
#                 (lifebase_bib_df['Lifebase'] == 'Valgrisenche IN') ) &
#                 (lifebase_bib_df['PK']== bib)]
#     new_bib_list.append(df)
    
# df= pd.concat(new_bib_list)

# pk_unique = list(df['PK'].unique())

# for pk in pk_unique:
#     print(lifebase_bib_df[['PK', 'Wave', 'Lifebase', 'Duration_hours']][(lifebase_bib_df['PK'] == pk)  ], '\n', '\n',)
#     print(TOR330_dem[['PK', 'Finish Category', 'Duration_hours']][(TOR330_dem['PK'] == pk)  ], '\n', '*'*40,  '\n',)

In [ ]:
# # reading in Raw Data
# races = ['TOR330']
# years = [ 
# #     '2021',
# #         '2022',
# #          '2023', 
#     '2024'
#         ]

# TORX_df = {}

# for race in races:
#     for year in years:
#         df = pd.read_excel(f'{race} Data/1. 100x100trail/{race}_{year}.xlsx',
#                                  dtype={'Start Date': 'string',
#                                         'Year': 'string'})
#         print(f'{race}_{year} {df.shape}')
#         # Store the DataFrame in the dictionary with a key like 'TOR330_2021'
#         TORX_df[f'{race}_{year}'] = df
#     print('*'*50)
    
# TORX_df_concat = pd.concat(TORX_df)
# TOR330 = TORX_df_concat[TORX_df_concat['Year'] == year]

In [ ]:
# sub_TOR330_dem_bib_list = list(TOR330_dem['PK'][TOR330_dem['Finish Category'] == 'Sub-90'].unique())

# new_bib_list = []
# for bib in sub_TOR330_dem_bib_list:
#     df = lifebase_bib_df[
#                 ((lifebase_bib_df['Duration_hours']> 8) &
#                 (lifebase_bib_df['Lifebase'] == 'Valgrisenche OUT') ) &
#                 (lifebase_bib_df['PK']== bib)]
#     new_bib_list.append(df)
    
# df= pd.concat(new_bib_list)

# pk_unique = list(df['PK'].unique())

# for pk in pk_unique:
#     print(lifebase_bib_df[['PK', 'Wave', 'Lifebase', 'Duration_hours']][(lifebase_bib_df['PK'] == pk)  ], '\n', '\n',)
#     print(TOR330_dem[['PK', 'Finish Category', 'Duration_hours']][(TOR330_dem['PK'] == pk)  ], '\n', '*'*40,  '\n',)